# LANL 3SBB — model vs experiment, one case at a time

For every experimental case the notebook renders

1. a 3D view of the building with the damage location highlighted, and
2. a 3×3 grid of the 9 accelerometer FRFs comparing the experimental
   median (blue) to the reduced-order model (red dashed) for that case.

Cases without a model counterpart (e.g. Mass-only configurations) are
shown with the experimental curve only; the model panel is left blank.


## 0. Colab bootstrap

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO   = 'https://github.com/grcarmenaty/PhD_LANL.git'
BRANCH = 'claude/fix-amplitudes-sensor-comparison-nzoFm'
WORK   = Path('/content/PhD_LANL')


def _safe_chdir(path):
    """chdir to *path*, falling back to / if the current cwd was deleted."""
    try:
        os.getcwd()
    except (FileNotFoundError, OSError):
        os.chdir('/')
    os.chdir(path)


def _run(cmd, **kw):
    """Run *cmd*, capture stdout/stderr, surface them on failure."""
    res = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if res.returncode != 0:
        print(f'\n--- command failed: {" ".join(map(str, cmd))}', file=sys.stderr)
        if res.stdout: print(res.stdout, file=sys.stderr)
        if res.stderr: print(res.stderr, file=sys.stderr)
    return res


def bootstrap():
    # If a previous run left the kernel cwd pointing at a directory that has
    # since been removed, every subprocess fails before it starts with
    # "Unable to read current working directory".  Pin cwd to a known-good
    # parent first.
    _safe_chdir('/content')

    if WORK.exists():
        if (WORK / '.git').exists():
            _run(['git', '-C', str(WORK), 'fetch', 'origin', BRANCH])
            _run(['git', '-C', str(WORK), 'checkout', BRANCH])
            _run(['git', '-C', str(WORK), 'reset', '--hard',
                  f'origin/{BRANCH}'])
        return

    res = _run(['git', 'clone', REPO, str(WORK)])
    if res.returncode != 0:
        res = _run(['git', 'clone', '--single-branch',
                    '--branch', BRANCH, REPO, str(WORK)])
    if res.returncode != 0:
        raise RuntimeError(
            f'git clone of {REPO} failed with exit {res.returncode}; see '
            f'stderr above.  Try a fresh Colab runtime, or run '
            f'`!rm -rf {WORK}` and re-execute this cell.'
        )
    _run(['git', '-C', str(WORK), 'checkout', BRANCH])


if 'google.colab' in sys.modules:
    bootstrap()
    _safe_chdir(WORK)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'h5py', 'numpy', 'scipy', 'matplotlib'], check=False)

print('cwd:', os.getcwd())
for f in ('median_frfs.h5', 'synthetic_frfs.h5'):
    p = Path(f)
    print(f'  {f}: {"OK" if p.exists() else "MISSING"}'
          + (f'  ({p.stat().st_size/1e6:.2f} MB)' if p.exists() else ''))

## 1. Imports and data

In [ ]:
import re
import numpy as np
import h5py
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from pathlib import Path

matplotlib.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 9,
})

EXP_H5 = Path('median_frfs.h5')
SYN_H5 = Path('synthetic_frfs.h5')


In [ ]:
# ── Experimental medians ───────────────────────────────────────────────────
with h5py.File(EXP_H5, 'r') as f:
    exp_freq    = f['freq'][:]
    median_frf  = f['median_frf'][:]                       # (61, 1601, 9)
    exp_cnames  = [c.decode() for c in f['case_names'][:]]
    exp_counts  = f['case_counts'][:]
exp_idx_of = {n: i for i, n in enumerate(exp_cnames)}
print(f'Experimental: {len(exp_cnames)} cases, {median_frf.shape}, '
      f'{exp_freq[0]:.2f}-{exp_freq[-1]:.1f} Hz')


In [ ]:
# ── Synthetic FRFs (model output) ──────────────────────────────────────────
with h5py.File(SYN_H5, 'r') as f:
    raw        = f['frfs'][:]
    syn_freqs  = f['freqs'][:]
    syn_cnames = [n.decode() if isinstance(n, bytes) else str(n)
                  for n in f['case_names'][:]]
    units      = str(f.attrs.get('units', '(m/s^2)/N')).lower().replace(' ', '')

# Auto-rescale based on the file's own units attribute (no more blind /1000)
if 'mm' in units:    scale = 1e-3
elif 'cm' in units:  scale = 1e-2
else:                scale = 1.0
syn_data = raw * scale
print(f'Synthetic: {len(syn_cnames)} model cases, units={units}, scale={scale:g}')

# If the synthetic file is the old 17-case version (Colab cached an earlier
# clone) the rest of the notebook will KeyError on case names.  Detect and
# fail early with an actionable message.
if len(syn_cnames) < len(exp_cnames):
    raise RuntimeError(
        f'synthetic_frfs.h5 has only {len(syn_cnames)} cases but the '
        f'experimental file has {len(exp_cnames)}.  This usually means a '
        f'cached Colab clone — re-run the bootstrap cell (it now does a '
        f'`git fetch && git reset --hard origin/{BRANCH}`), or delete '
        f'/content/PhD_LANL and re-run from scratch.'
    )

## 2. Damage parser (used by the 3D renderer)

`synthetic_frfs.h5` is now generated 1:1 against `median_frfs.h5` —
every experimental case has a synthetic counterpart at the same case
name and index — so no name-matching logic is needed.  The parser
below is used only by the 3D renderer in section 3 to decide which
storeys/columns/plates to highlight.

In [ ]:
# Channel layout (HDF5 column order)
SIG_NAMES = [2, 5, 6, 7, 8, 11, 12, 13, 14]


def parse_damage(name: str):
    """Return a list of (kind, storey, info) tuples for the 3D renderer.

    Storeys are 0-based (storey 0 = base→floor 1).
    plate index 0..3 with 0=base plate, 3=roof.
    """
    s = name.lower()
    if 'pristine' in s:
        return [('pristine',)]
    out = []

    # Bolt damage: D(X%) nBD / Damage (X%) nBD / nAD
    pct_iter = re.finditer(r'(?:d|damage)\s*\(?\s*(\d+)\s*%?\s*\)?', s)
    pcts = [int(m.group(1)) for m in pct_iter]
    bd = re.findall(r'(\d)\s*bd', s)
    ad = re.findall(r'(\d)\s*ad', s)
    if pcts and (bd or ad):
        all_st = bd + ad
        if len(pcts) == 1:
            pcts = pcts * len(all_st)
        for i, st in enumerate(all_st):
            kind = 'bolt_below' if i < len(bd) else 'bolt_above'
            out.append((kind, int(st) - 1, {'pct': pcts[min(i, len(pcts)-1)]}))

    # Cracks
    for m in re.finditer(
            r'crack\s*(\d+)\s*mm.*?(\d)\s*[ab]d|crack\s*(\d)\s*[ab]d\s*(\d+)\s*mm', s):
        if m.group(1):
            size = int(m.group(1)); st = int(m.group(2))
        else:
            st = int(m.group(3)); size = int(m.group(4))
        out.append(('crack', st - 1, {'size_mm': size}))

    # Holes
    for m in re.finditer(r'hole\s*(\d+)\s*mm.*?(\d)\s*[ab]d', s):
        out.append(('hole', int(m.group(2)) - 1, {'size_mm': int(m.group(1))}))

    # Added masses
    if 'mass' in s:
        if 'base'         in s: out.append(('mass', 0, {}))
        if 'first floor'  in s or 'mass 1f' in s: out.append(('mass', 1, {}))
        if 'second floor' in s: out.append(('mass', 2, {}))
        if 'third floor'  in s: out.append(('mass', 3, {}))

    return out or [('unknown',)]


# Sanity check — every experimental case must be present in the synthetic file
missing = [n for n in exp_cnames if n not in syn_cnames]
print(f'Missing synthetic counterparts: {len(missing)}  (expected: 0)')
if missing:
    for n in missing[:10]: print(f'  - {n}')

## 3. 3D rendering of the building with damage indicators

In [ ]:
# Geometry constants (same as model_3sbb.py)
PL  = 0.305         # plate side length         (m)
PT  = 0.0254        # plate thickness           (m)
CLX = 0.0254        # column wide dimension     (m)
CLY = 0.0064        # column thin dimension     (m)
CGAP = 0.0005       # column-to-plate gap       (m)
SH  = PT + 0.1524   # storey height             (m)

z_pl = [k * SH + PT/2 for k in range(4)]   # 4 plate centre z's

xl = CLX / 2
xh = PL - xl
cx = PL / 2

# 9 sensor positions in HDF5 channel order
SENSOR_POS = [
    (xl, PL, z_pl[0]),  # S2  base +Y xl  (inverted polarity in exp)
    (xl, PL, z_pl[3]),  # S5  fl3  +Y xl
    (xl, PL, z_pl[2]),  # S6  fl2  +Y xl
    (xl, PL, z_pl[1]),  # S7  fl1  +Y xl
    (xh, PL, z_pl[0]),  # S8  base +Y xh
    (xh, PL, z_pl[3]),  # S11 fl3  +Y xh
    (xh, PL, z_pl[2]),  # S12 fl2  +Y xh
    (xh, PL, z_pl[1]),  # S13 fl1  +Y xh
    (cx, 0., z_pl[0]),  # S14 base -Y cx
]
SHAKER_POS = (cx, 0., z_pl[0])

# Column attachment points (4 corners) in plate xy plane
COL_XY = [
    (xl, -CLY/2 - CGAP),
    (xh, -CLY/2 - CGAP),
    (xl, PL + CLY/2 + CGAP),
    (xh, PL + CLY/2 + CGAP),
]


def _box_faces(x0, x1, y0, y1, z0, z1):
    v = [(x0,y0,z0),(x1,y0,z0),(x1,y1,z0),(x0,y1,z0),
         (x0,y0,z1),(x1,y0,z1),(x1,y1,z1),(x0,y1,z1)]
    return [[v[i] for i in idx]
            for idx in [(0,1,2,3),(4,5,6,7),(0,1,5,4),(3,2,6,7),(0,3,7,4),(1,2,6,5)]]


def render_building(ax, case_name):
    """Render the 3SBB on *ax* (must be a 3D axes) with damage indicators."""
    spec = parse_damage(case_name)

    # Highlight maps: storey index → colour for column highlight
    storey_colour = {}
    plate_mass    = set()
    column_marks  = []     # list of (storey, kind, info)

    for tag in spec:
        if tag[0] == 'pristine':
            pass
        elif tag[0].startswith('bolt'):
            st = tag[1]
            pct = tag[2].get('pct', 0)
            storey_colour[st] = ('bolt', pct)
        elif tag[0] in ('crack', 'hole'):
            column_marks.append(tag)
            storey_colour.setdefault(tag[1], (tag[0], tag[2].get('size_mm', 0)))
        elif tag[0] == 'mass':
            plate_mass.add(tag[1])

    # Plates
    for k, zc in enumerate(z_pl):
        zb, zt = zc - PT/2, zc + PT/2
        face_clr = 'silver'
        ax.add_collection3d(Poly3DCollection(
            _box_faces(0, PL, 0, PL, zb, zt),
            alpha=0.20, facecolor=face_clr, edgecolor='gray', lw=0.4))
        ax.text(PL/2, PL/2, zt + 0.005, ['Base','Floor 1','Floor 2','Floor 3'][k],
                ha='center', va='bottom', fontsize=7, color='dimgray')

    # Columns (4 corners x 3 storeys)
    for cxc, cyc in COL_XY:
        for st in range(3):
            col_kind = storey_colour.get(st)
            if col_kind is None:
                colour, lw = 'steelblue', 3.0
            elif col_kind[0] == 'bolt':
                # Lighter red for milder damage
                pct = col_kind[1]
                t = min(1.0, pct / 85.0)
                colour = (0.65 + 0.35*t, 0.15 - 0.10*t, 0.15)
                lw = 4.0
            elif col_kind[0] == 'crack':
                colour, lw = 'orange', 4.0
            elif col_kind[0] == 'hole':
                colour, lw = 'black', 4.0
            else:
                colour, lw = 'steelblue', 3.0
            ax.plot([cxc, cxc], [cyc, cyc],
                    [z_pl[st], z_pl[st+1]],
                    color=colour, lw=lw, solid_capstyle='round')

    # Localised crack / hole markers (small spheres at column mid-height)
    for kind, st, info in column_marks:
        zc = (z_pl[st] + z_pl[st+1]) / 2.0
        for cxc, cyc in COL_XY[:1]:        # mark on first corner column
            ax.scatter([cxc], [cyc], [zc],
                       s=120 if kind == 'hole' else 90,
                       c='black' if kind == 'hole' else 'orange',
                       marker='o' if kind == 'hole' else 's',
                       edgecolor='red', zorder=8)
            ax.text(cxc, cyc - 0.04, zc,
                    f"{kind} {info.get('size_mm','?')}mm",
                    fontsize=6, color='red')

    # Mass blocks on top of plates
    for k in plate_mass:
        zc = z_pl[k] + PT/2
        m_h = 0.022
        ax.add_collection3d(Poly3DCollection(
            _box_faces(PL/2 - 0.04, PL/2 + 0.04,
                       PL/2 - 0.04, PL/2 + 0.04,
                       zc, zc + m_h),
            alpha=0.85, facecolor='goldenrod', edgecolor='black', lw=0.6))
        ax.text(PL/2, PL/2, zc + m_h + 0.005, 'mass',
                ha='center', fontsize=6, color='darkgoldenrod')

    # Sensors
    for (sx, sy, sz), nm in zip(SENSOR_POS, SIG_NAMES):
        ax.scatter([sx], [sy], [sz], s=42, c='tab:blue',
                   marker='^', zorder=6, depthshade=False)
        ax.text(sx, sy + 0.015, sz + 0.008, f'S{nm}',
                fontsize=6, color='tab:blue')

    # Shaker
    sx, sy, sz = SHAKER_POS
    ax.scatter([sx], [sy], [sz], s=180, c='darkorange',
               marker='*', zorder=7, depthshade=False)
    ax.text(sx, sy - 0.04, sz + 0.005, 'shaker',
            ha='center', fontsize=6, color='darkorange')

    ax.set_xlim(0, PL); ax.set_ylim(-0.05, PL + 0.05)
    ax.set_zlim(0, z_pl[3] + 0.04)
    ax.set_xlabel('X', labelpad=2, fontsize=7)
    ax.set_ylabel('Y', labelpad=2, fontsize=7)
    ax.set_zlabel('Z', labelpad=2, fontsize=7)
    ax.tick_params(labelsize=6, pad=0)
    ax.view_init(elev=20, azim=-55)


## 4. Per-case overview

Each row below = one experimental case. Left: 3D model with damage
indicators. Right: 3×3 grid of |H| for all 9 sensors, experimental
median (blue) over the model (red dashed) — the model line is absent
for cases without a synthetic counterpart (e.g. Mass-only).


In [ ]:
# 3x3 sensor grid layout: rows = floor (top→bottom), cols = column position
GRID = [
    [1, 5, None],     # floor 3:  S5 (xl), S11 (xh)
    [2, 6, None],     # floor 2:  S6 (xl), S12 (xh)
    [3, 7, None],     # floor 1:  S7 (xl), S13 (xh)
    [0, 4,    8],     # base:     S2 (xl), S8 (xh), S14 (cx -Y)
]
ROW_LABEL = ['Floor 3', 'Floor 2', 'Floor 1', 'Base plate']

syn_idx_of = {n: i for i, n in enumerate(syn_cnames)}


def render_case(case_name):
    si      = syn_idx_of[case_name]
    H_exp   = median_frf[exp_idx_of[case_name]]            # (1601, 9)
    H_mod   = syn_data[si]                                  # (1601, 9)
    n_meas  = int(exp_counts[exp_idx_of[case_name]])

    fig = plt.figure(figsize=(15, 7.5))
    gs  = GridSpec(4, 7, width_ratios=[1.4, 1.4, 1.4, 0.05, 1, 1, 1],
                   wspace=0.35, hspace=0.35)
    ax3d = fig.add_subplot(gs[:, :3], projection='3d')
    render_building(ax3d, case_name)
    ax3d.set_title(f'{case_name}\n({n_meas} measurements aggregated)',
                   fontsize=10)

    for r, row in enumerate(GRID):
        for c, ch in enumerate(row):
            ax = fig.add_subplot(gs[r, 4 + c])
            if ch is None:
                ax.set_visible(False); continue
            ax.semilogy(exp_freq, np.abs(H_exp[:, ch]),
                        color='tab:blue', lw=1.2, label='exp median')
            ax.semilogy(syn_freqs, np.abs(H_mod[:, ch]),
                        'r--', lw=1.0, label='model')
            ax.set_xlim(0, 100); ax.set_ylim(1e-3, 1e1)
            ax.grid(True, which='both', ls=':', alpha=0.3)
            ax.tick_params(labelsize=6)
            ax.set_title(f'S{SIG_NAMES[ch]} — {ROW_LABEL[r]}',
                         fontsize=8, pad=2)
            if r == 3: ax.set_xlabel('Hz', fontsize=7)
            if c == 0: ax.set_ylabel('|H| (m/s²)/N', fontsize=7)
            if r == 0 and c == 0:
                ax.legend(fontsize=6, loc='lower right')
    plt.show()


# Iterate over every experimental case.
CASES_TO_PLOT = list(exp_cnames)

print(f'Rendering {len(CASES_TO_PLOT)} cases ...\n')
for nm in CASES_TO_PLOT:
    render_case(nm)

## 5. CFDAC and SCI per case

Same layout as section 4 (3D model on the left), now with the two
CFDAC matrices on the right.  The Complex Frequency Domain Assurance
Criterion couples every pair of frequencies via the cross-correlation
of the FRF row vectors across the 9 sensors:

$$\\mathrm{CFDAC}_{ij}\\;=\\;\\frac{\\bigl|\\mathbf{H}(f_i)^*\\,\\mathbf{H}(f_j)\\bigr|^2}{\\bigl(\\mathbf{H}(f_i)^*\\,\\mathbf{H}(f_i)\\bigr)\\,\\bigl(\\mathbf{H}(f_j)^*\\,\\mathbf{H}(f_j)\\bigr)}\\;\\in\\;[0,1]$$

The **Squared Correlation Index (SCI)** between the experimental and
synthetic CFDAC matrices is the squared Pearson correlation of their
flattened entries — a single scalar in `[0, 1]` that summarises how
similar the two CFDAC patterns are.  It is shown in the 3D-plot title
for each case.

Analysis is restricted to 5–80 Hz (the IQS sine-sweep band) and
sub-sampled to 1 Hz steps (76 frequencies) so the 76² CFDAC matrices
remain easy to plot.

In [ ]:
# ── CFDAC + SCI helpers ─────────────────────────────────────────────────
F_LO, F_HI = 5.0, 80.0
DF_CFDAC   = 1.0          # frequency step inside the CFDAC band

def _band_indices(freq, f_lo=F_LO, f_hi=F_HI, step=DF_CFDAC):
    """Sub-sample *freq* to ~step Hz over [f_lo, f_hi]."""
    target = np.arange(f_lo, f_hi + 1e-9, step)
    return np.array([int(np.argmin(np.abs(freq - t))) for t in target])


def cfdac(H_band):
    """CFDAC matrix from a (N_freq, N_sensors) complex FRF block."""
    inner = H_band.conj() @ H_band.T          # (N, N) complex
    diag  = np.real(np.diag(inner)).copy()
    diag[diag < 1e-30] = 1e-30
    norm  = np.sqrt(np.outer(diag, diag))
    return (np.abs(inner) ** 2) / (norm ** 2)


def sci(C1, C2):
    """Squared Pearson correlation of two CFDAC matrices."""
    a = C1.ravel(); b = C2.ravel()
    a = a - a.mean(); b = b - b.mean()
    denom = np.sqrt((a @ a) * (b @ b))
    return float((a @ b) ** 2 / denom ** 2) if denom > 0 else 0.0


# Pre-compute experimental and synthetic CFDACs for every case
exp_band_idx = _band_indices(exp_freq)
syn_band_idx = _band_indices(syn_freqs)
band_freqs   = exp_freq[exp_band_idx]
print(f'CFDAC band: {band_freqs[0]:.1f}–{band_freqs[-1]:.1f} Hz, '
      f'{len(band_freqs)} frequencies')

cfdac_exp = np.empty((len(exp_cnames), len(band_freqs), len(band_freqs)),
                      dtype=np.float32)
cfdac_syn = np.empty_like(cfdac_exp)
sci_per_case = np.empty(len(exp_cnames), dtype=np.float64)
for i, n in enumerate(exp_cnames):
    He = median_frf[i][exp_band_idx]                 # (Nb, 9) complex
    Hs = syn_data[syn_idx_of[n]][syn_band_idx]       # (Nb, 9) complex
    cfdac_exp[i] = cfdac(He).astype(np.float32)
    cfdac_syn[i] = cfdac(Hs).astype(np.float32)
    sci_per_case[i] = sci(cfdac_exp[i], cfdac_syn[i])
print('Done; mean SCI =', float(sci_per_case.mean()))

In [ ]:
# ── Per-case 3D model + CFDAC pair ─────────────────────────────────────
# Mirrors the FRF section layout: 3D building on the left, then the two
# CFDAC matrices side by side (experiment, model).  SCI value is shown
# in the title of the 3D plot so each case has a single scoreboard line.
def render_cfdac_case(case_name):
    i  = exp_cnames.index(case_name)
    Ce = cfdac_exp[i]
    Cs = cfdac_syn[i]
    s  = sci_per_case[i]
    n_meas = int(exp_counts[i])

    fig = plt.figure(figsize=(15, 5.4))
    gs  = GridSpec(1, 4,
                   width_ratios=[1.6, 1.0, 1.0, 0.04],
                   wspace=0.30)
    ax3d = fig.add_subplot(gs[0, 0], projection='3d')
    render_building(ax3d, case_name)
    ax3d.set_title(f'{case_name}\n({n_meas} measurements)   SCI = {s:.3f}',
                   fontsize=10)

    extent = [band_freqs[0], band_freqs[-1], band_freqs[-1], band_freqs[0]]
    ax_e = fig.add_subplot(gs[0, 1])
    ax_e.imshow(Ce, vmin=0, vmax=1, extent=extent,
                cmap='magma', aspect='equal')
    ax_e.set_title('CFDAC — experiment', fontsize=9)
    ax_e.set_xlabel('f_j (Hz)', fontsize=8)
    ax_e.set_ylabel('f_i (Hz)', fontsize=8)
    ax_e.tick_params(labelsize=7)

    ax_s = fig.add_subplot(gs[0, 2])
    im_s = ax_s.imshow(Cs, vmin=0, vmax=1, extent=extent,
                       cmap='magma', aspect='equal')
    ax_s.set_title('CFDAC — model', fontsize=9)
    ax_s.set_xlabel('f_j (Hz)', fontsize=8)
    ax_s.tick_params(labelsize=7)
    ax_s.set_yticklabels([])

    cax = fig.add_subplot(gs[0, 3])
    fig.colorbar(im_s, cax=cax)
    cax.tick_params(labelsize=7)
    plt.show()


for nm in CASES_TO_PLOT:
    render_cfdac_case(nm)

## 6. SCI summary across all cases

In [ ]:
# Bar chart of SCI per case, sorted ascending so the worst-fit cases sit
# on the left.  A horizontal line at the mean gives a quick reference.
order   = np.argsort(sci_per_case)
labels  = [exp_cnames[i] for i in order]
values  = sci_per_case[order]
mean_   = float(values.mean())

fig, ax = plt.subplots(figsize=(14, max(4, 0.18 * len(labels))))
bars = ax.barh(np.arange(len(labels)), values, color='steelblue', alpha=0.85)
ax.axvline(mean_, color='crimson', lw=1.0, ls='--',
           label=f'mean SCI = {mean_:.3f}')
ax.set_yticks(np.arange(len(labels)))
ax.set_yticklabels(labels, fontsize=6)
ax.set_xlabel('SCI  (1 = perfect CFDAC match, 0 = uncorrelated)')
ax.set_xlim(0, 1)
ax.grid(axis='x', alpha=0.3)
ax.legend(loc='lower right', fontsize=8)
ax.set_title('CFDAC similarity (SCI) — model vs experiment, all 61 cases',
             fontsize=11)
plt.tight_layout()
plt.show()

print(f'\nSummary:')
print(f'  cases : {len(values)}')
print(f'  mean  : {values.mean():.3f}')
print(f'  median: {float(np.median(values)):.3f}')
print(f'  min   : {values.min():.3f}  ({labels[0]})')
print(f'  max   : {values.max():.3f}  ({labels[-1]})')

## 7. Status

**Where we are**

Mean SCI **0.925**, median **0.959** (band 5–100 Hz, all 61 cases).
**39 of 61 cases above SCI = 0.95**, **49 of 61 above 0.90**.
Mode 1/2/3 frequencies pinned to 21.2 / 50.2 / 68.2 Hz vs target
20.94 / 49.94 / 68.19 (within 0.5 %).  Mode 1/2/3 amplitudes match
experiment to 99–100 % after the damping fit, the floor-3 4th peak
sits where the experiment shows it, and the asymmetric damage cases
(`AD` vs `BD`, `1BD + 2BD`) now diverge correctly.

**This iteration's three architectural changes**

1. **Per-column-end JSR with the asymmetric semi-rigid formula.**
   `BuildingGeometry.joint_stiffness_per_end` is a `(n_stories, 4, 2)`
   array — last axis `[bottom, top]`.  Where finite it replaces the
   scalar JSR for that specific (storey, corner, end), and the column
   stiffness is then computed from the asymmetric formula
   $$k_\\mathrm{eff} = k_\\mathrm{ff}\\,
       \\frac{J_t J_b + J_t + J_b}{J_t J_b + 4(J_t+J_b) + 12}$$
   (derived in `reduced_model_semirigid._semirigid_factor`; reduces
   to $J/(J+6)$ for symmetric ends, $1$ for both rigid, $0$ for
   either pinned, and $(J+1)/(J+4)$ when one end is rigid — matches
   the canonical fixed-pinned column result of $3EI/L^3$).
   `damage_scenarios` now switches bolt damage from `column_factor`
   reduction to per-end JSR reduction (`BD` → bottom-end, `AD` →
   top-end), so single-end damage produces less stiffness loss than
   damage at both ends — exactly what `D(X%) 1AD + D(X%) 1BD` vs
   `D(X%) 1BD` shows.

2. **Floor-3 4th peak frequency raised above 100 Hz.**
   `plate_flex_freq` is now hard-bounded to `[110, 150]` Hz so the
   tuned-attachment peak sits where the experimental rise actually
   leads (above the data band) and only the rising left flank is
   visible in 95–100 Hz.

3. **Per-case parameter overrides** (`case_overrides.py`).
   The generic damage parser still produces every geometry from the
   same global calibration, but `apply_overrides(g, case_name)`
   layers small case-specific multiplicative tweaks on top.  Empty
   by default — populated by `calibrate_per_case.py`, which grid-
   searches a few override-key combinations per problem case
   (`D(85%) 1BD + D(85%) 2BD` jumped from SCI 0.79 → 0.97 from this
   alone).

**Scoreboard**

| metric   | initial | + rigid | + SCI v1 | + SCI v2 + damping | + flex + direct C | + per-end JSR + overrides |
|----------|--------:|--------:|---------:|-------------------:|------------------:|--------------------------:|
| mean SCI | 0.461   | 0.739   | 0.891    | 0.921 *(5-80)*     | 0.899 *(5-100)*   | **0.925**                 |
| median   | 0.474   | 0.770   | 0.923    | 0.961              | 0.937             | **0.959**                 |
| ≥ 0.95   | —       | —       | —        | —                  | 6 / 61            | **39 / 61**               |
| ≥ 0.90   | —       | —       | —        | —                  | —                 | **49 / 61**               |

**Bottom 5 today**

| case                                     | SCI   | what it needs                        |
|------------------------------------------|------:|---------------------------------------|
| Pristine (26/1/2021)                     | 0.580 | per-session calibration (different setup) |
| Pristine (27/1/2021)                     | 0.580 | same                                 |
| `D(85%) 2BD`                             | 0.595 | wider override grid, possibly per-mass tweak |
| `D(11%) 1BD`                             | 0.762 | per-storey JSR ratio (currently uses one ratio per damage %) |
| `D(85%) 2BD + D(85%) 2AD + Mass Base`    | 0.861 | mass + 2BD interaction tweak         |

The `Pristine (26/27 Jan 2021)` rows are a measurement-side issue:
their experimental CFDACs visibly differ from every other Pristine
variant, so no single global calibration can fit both the canonical
and outlier sessions.

**Direct-inversion path (still in place from the previous iteration)**

`compute_frf_matrix` continues to dispatch to `compute_frf_direct`
when `geom.grounded_oscillators` carries `damp_coupling > 0` or any
explicit `dashpot_couplings` are configured.  Off the calibrated
path the modal-superposition and direct-inversion outputs match to
ratio 1.000 on every sensor at every frequency.

**What still moves the needle**

1. **Plate / column discretisation** — split each plate into two
   stacked sub-masses, or each column into two segments with an
   internal mid-column mass.  Adds Y-DOFs that are part of the
   primary eigenstructure (no tuned-attachment anti-resonance), so
   the 100+ Hz region can finally carry a real Y-mode peak.
2. **Per-storey damage JSR ratios.**  `_BOLT_JSR_RATIO` is one table
   for all storeys; `D(11%) 1BD` (SCI 0.76) suggests storey 1 needs
   a different ratio than storey 2 or 3.
3. **Damage-physics submodels** (Coulomb-friction bolt contact,
   Castigliano crack hinge, rotational inertia on added masses).
4. **3-D continuum FE digital twin** for ground-truth
   reduced-order error bounds.
5. **Bayesian calibration** on `(JSR_i, plate_extra_mass,
   plate_flex_freq, plate_flex_mass, override_ratios)` — needed for
   damage-detection where the FRF change has to exceed the
   parameter-uncertainty FRF cone.